# 12: The PyTorch Training Loop

## The Standard ML Workflow

Every ML project in PyTorch follows the same pattern. Master this and you can train any model!

```
for epoch in epochs:
    for batch in dataloader:
        1. Forward pass (compute predictions)
        2. Compute loss
        3. Backward pass (compute gradients)
        4. Update weights
        5. Zero gradients
```

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(42)

print("Ready to learn the training loop! 🔄")

## 1. Building Models with nn.Module

In [ ]:
# Simple neural network for classification
class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        x = self.layer1(x)
        x = self.activation(x)
        x = self.layer2(x)
        return x

model = SimpleNN(input_size=2, hidden_size=8, output_size=1)
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params}")

## 2. DataLoaders: Feeding Data in Batches

In [ ]:
# Generate XOR-like data
np.random.seed(42)
n_samples = 1000

X_np = np.vstack([
    np.random.randn(n_samples//4, 2) * 0.5 + [0, 0],
    np.random.randn(n_samples//4, 2) * 0.5 + [1, 1],
    np.random.randn(n_samples//4, 2) * 0.5 + [0, 1],
    np.random.randn(n_samples//4, 2) * 0.5 + [1, 0],
])
y_np = np.array([0]*(n_samples//4) + [0]*(n_samples//4) + [1]*(n_samples//4) + [1]*(n_samples//4))

X = torch.FloatTensor(X_np)
y = torch.FloatTensor(y_np).reshape(-1, 1)

dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

print(f"Dataset size: {len(dataset)}")
print(f"Number of batches: {len(dataloader)}")

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X_np[y_np==0, 0], X_np[y_np==0, 1], c='red', alpha=0.5, label='Class 0')
plt.scatter(X_np[y_np==1, 0], X_np[y_np==1, 1], c='blue', alpha=0.5, label='Class 1')
plt.xlabel('X1')
plt.ylabel('X2')
plt.title('XOR-like Classification Problem')
plt.legend()
plt.show()

## 3. Loss Functions and Optimizers

In [ ]:
print("Common Loss Functions:")
print("-" * 50)
print("Binary: BCELoss, BCEWithLogitsLoss")
print("Multi-class: CrossEntropyLoss, NLLLoss")
print("Regression: MSELoss, L1Loss")

print("\nCommon Optimizers:")
print("-" * 50)
print("SGD: Simple, needs tuning")
print("SGD+momentum: Faster convergence")
print("Adam: Adaptive, works well by default")
print("AdamW: Adam + proper weight decay")

## 4. The Complete Training Loop

In [ ]:
model = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

history = {'loss': [], 'accuracy': []}
n_epochs = 50

print("Training...")
print("=" * 50)

for epoch in range(n_epochs):
    epoch_loss = 0
    correct = 0
    total = 0
    
    for batch_x, batch_y in dataloader:
        # 1. Forward pass
        outputs = model(batch_x)
        
        # 2. Compute loss
        loss = criterion(outputs, batch_y)
        
        # 3. Backward pass
        loss.backward()
        
        # 4. Update weights
        optimizer.step()
        
        # 5. Zero gradients
        optimizer.zero_grad()
        
        epoch_loss += loss.item()
        predictions = (torch.sigmoid(outputs) > 0.5).float()
        correct += (predictions == batch_y).sum().item()
        total += batch_y.size(0)
    
    avg_loss = epoch_loss / len(dataloader)
    accuracy = correct / total
    history['loss'].append(avg_loss)
    history['accuracy'].append(accuracy)
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d}: Loss = {avg_loss:.4f}, Accuracy = {accuracy:.2%}")

print(f"\n✅ Final: Loss = {history['loss'][-1]:.4f}, Accuracy = {history['accuracy'][-1]:.2%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['loss'])
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')

axes[1].plot(history['accuracy'])
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training Accuracy')

plt.tight_layout()
plt.show()

## 5. Visualize Decision Boundary

In [ ]:
# Set to inference mode
model.train(False)

xx, yy = np.meshgrid(np.linspace(-1, 2, 200), np.linspace(-1, 2, 200))
grid = torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()])

with torch.no_grad():
    Z = torch.sigmoid(model(grid)).numpy().reshape(xx.shape)

plt.figure(figsize=(10, 8))
plt.contourf(xx, yy, Z, levels=np.linspace(0, 1, 11), cmap='RdBu', alpha=0.6)
plt.colorbar(label='P(class=1)')
plt.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)

plt.scatter(X_np[y_np==0, 0], X_np[y_np==0, 1], c='red', alpha=0.5, s=20)
plt.scatter(X_np[y_np==1, 0], X_np[y_np==1, 1], c='blue', alpha=0.5, s=20)

plt.xlabel('X1')
plt.ylabel('X2')
plt.title('Learned Decision Boundary')
plt.show()

## 6. Training vs Inference Mode

Some layers behave differently during training vs inference:
- **Dropout**: drops neurons during training only
- **BatchNorm**: uses batch stats during training, running stats during inference

In [ ]:
model_with_dropout = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(16, 1)
)

# Training mode (default)
model_with_dropout.train(True)
print(f"Training mode: {model_with_dropout.training}")

# Inference mode
model_with_dropout.train(False)
print(f"Inference mode: {model_with_dropout.training}")

print("\n⚠️  Always set model.train(False) before inference!")
print("   And use torch.no_grad() to disable gradient computation.")

## 7. Saving and Loading Models

In [ ]:
import os

os.makedirs('../../data', exist_ok=True)

# Save model state dict (recommended approach)
torch.save(model.state_dict(), '../../data/model_weights.pth')
print("Saved model weights")

# Load into a new model
new_model = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)
new_model.load_state_dict(torch.load('../../data/model_weights.pth'))
new_model.train(False)
print("Loaded model weights")

# Verify
with torch.no_grad():
    test_input = torch.tensor([[0.5, 0.5]])
    original_output = model(test_input)
    loaded_output = new_model(test_input)
    print(f"\nOriginal output: {original_output.item():.4f}")
    print(f"Loaded output: {loaded_output.item():.4f}")
    print("✅ Outputs match!")

## 📝 Check Your Understanding

1. What's the order of operations in a training step?
2. Why must we call `optimizer.zero_grad()`?
3. What's the difference between training and inference mode?
4. Why use `torch.no_grad()` during validation?
5. How do you save and load a trained model?

## 🎯 Summary

The PyTorch training loop:
1. **Forward pass**: `outputs = model(inputs)`
2. **Compute loss**: `loss = criterion(outputs, targets)`
3. **Backward pass**: `loss.backward()`
4. **Update weights**: `optimizer.step()`
5. **Zero gradients**: `optimizer.zero_grad()`

Key patterns:
- `model.train(True)` for training, `model.train(False)` for inference
- `torch.no_grad()` during evaluation
- `torch.save()` / `torch.load()` for persistence

**Next up**: Word Embeddings - giving words meaning! →